In [ ]:
import os
import json
import shutil
import cv2
import numpy as np

from typing import List

def process_landmark_annotations(
    json_path: str,
    output_dir: str,
    images_dir: str,
    normalize: bool = False
) -> None:
    """
    Process JSON facial landmark annotations and create organized .txt files for each image.

    Args:
        json_path (str): Path to the JSON file containing the landmark annotations.
        output_dir (str): Path to the output directory where 'images' and 'labels' folders will be created.
        images_dir (str): Path to the directory containing the original images.
        normalize (bool): If True, normalize landmark coordinates with respect to the image dimensions.

    Returns:
        None
    """
    # Define the ordered list of 72 landmarks
    landmarks_order = [
        f"face_contour_{i}" for i in range(1, 18)
    ] + [
        f"right_eyebrow_{i}" for i in range(18, 23)
    ] + [
        f"left_eyebrow_{i}" for i in range(23, 28)
    ] + [
        f"nose_bridge_{i}" for i in range(28, 32)
    ] + [
        f"nose_base_{i}" for i in range(32, 37)
    ] + [
        f"right_eye_{i}" for i in range(37, 43)
    ] + [
        f"left_eye_{i}" for i in range(43, 49)
    ] + [
        f"outer_lip_{i}" for i in range(49, 61)
    ] + [
        f"inner_lip_{i}" for i in range(61, 69)
    ] + [
        "under_lip_69", "upper_chin70", "left_chin_71", "right_chin_72"
    ]

    # Create output directories
    images_output_dir = os.path.join(output_dir, "images")
    labels_output_dir = os.path.join(output_dir, "labels")
    os.makedirs(images_output_dir, exist_ok=True)
    os.makedirs(labels_output_dir, exist_ok=True)

    # Load the JSON annotations
    with open(json_path, 'r') as f:
        annotations = json.load(f)

    for annotation in annotations:
        # Extract image information
        image_path = annotation["image"].split("?d=")[-1]  # Clean up image path
        image_name = os.path.basename(image_path)
        original_width = annotation["landmarks"][0]["original_width"]
        original_height = annotation["landmarks"][0]["original_height"]

        # Initialize landmarks with NaN for each image
        landmarks_data = {key: (np.nan, np.nan) for key in landmarks_order}

        # Fill in available landmark data
        for landmark in annotation["landmarks"]:
            label = landmark["keypointlabels"][0]
            x, y = landmark["x"], landmark["y"]

            # Step 1: Ensure desnormalization from 0-100 to absolute pixel
            if 0 <= x <= 100 and 0 <= y <= 100:
                # Rescale x and y from [0, 100] to [0, original_width] and [0, original_height]
                x = (x / 100) * original_width
                y = (y / 100) * original_height

            # Step 2: Normalize if requested
            if normalize:
                x /= original_width
                y /= original_height

            # Convert to integer if not normalized
            if not normalize:
                x = int(round(x))
                y = int(round(y))

            landmarks_data[label] = (x, y)

        # Create a single line for this instance
        instance_line = " ".join(
            f"{x:.6f} {y:.6f}" if normalize else f"{x} {y}"
            if not (np.isnan(x) or np.isnan(y)) else "nan nan"
            for x, y in landmarks_data.values()
        )

        # Write the annotation to the .txt file
        txt_path = os.path.join(labels_output_dir, f"{os.path.splitext(image_name)[0]}.txt")
        with open(txt_path, 'w') as txt_file:
            txt_file.write(instance_line + "\n")

        # Copy image to output folder
        src_image_path = os.path.join(images_dir, image_name)
        dst_image_path = os.path.join(images_output_dir, image_name)
        if os.path.exists(src_image_path):
            shutil.copy(src_image_path, dst_image_path)
        else:
            print(f"Warning: Image {src_image_path} not found, skipping.")

    print(f"Processing complete. Output saved to {output_dir}.")




# Example usage
process_landmark_annotations(
    json_path="/Users/jocareher/Downloads/babyface_72_311.json",
    output_dir="/Users/jocareher/Downloads/baby_face_72_311",
    images_dir="/Users/jocareher/Downloads/under_construction_dataset/Hospital_del_Mar",
    normalize=False
)


Processing complete. Output saved to /Users/jocareher/Downloads/baby_face_72_311.


In [5]:
def load_landmarks_from_txt(file_path):
    """
    Lee landmarks desde un archivo .txt.
    Cada línea contiene pares x, y en secuencia, algunos pueden ser NaN.
    Devuelve un array de shape (N, 2) con N landmarks. 
    Los que sean NaN se dejan como np.nan.
    """
    if not os.path.exists(file_path) or os.stat(file_path).st_size == 0:
        return None

    with open(file_path, 'r') as f:
        line = f.readline().strip()
    
    coords = list(map(float, line.split()))
    
    if len(coords) % 2 != 0:
        print(f"Advertencia: el archivo {file_path} no tiene un número par de valores.")
        return None

    landmarks = np.array(coords).reshape(-1, 2)
    return landmarks

def draw_landmarks_on_image(image, landmarks, color=(0, 0, 255)):
    """
    Dibuja los landmarks sobre la imagen, ignorando los que tengan NaN.
    El tamaño del punto (radio) se ajusta según el tamaño de la imagen.
    """
    # Calcula un radio proporcional al tamaño de la imagen
    h, w = image.shape[:2]
    # Por ejemplo, tomar 1% del lado más corto como radio
    radius = max(1, int(min(h, w) * 0.005))
    thickness = -1  # -1 para un círculo relleno

    for (x, y) in landmarks:
        if not (np.isnan(x) or np.isnan(y)):
            cv2.circle(image, (int(round(x)), int(round(y))), radius, color, thickness)
    return image

def overlay_landmarks_on_images(image_dir, txt_dir, output_dir):
    """
    Recorre un directorio con imágenes, busca sus landmarks en el directorio txt_dir,
    dibuja los landmarks sobre la imagen y guarda el resultado en output_dir.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])

    for img_file in image_files:
        base_name = os.path.splitext(img_file)[0]
        txt_file = base_name + '.txt'

        img_path = os.path.join(image_dir, img_file)
        txt_path = os.path.join(txt_dir, txt_file)

        if not os.path.exists(txt_path):
            print(f"No se encontró el archivo de landmarks para {img_file}, se omite.")
            continue

        # Cargar imagen
        image = cv2.imread(img_path)
        if image is None:
            print(f"No se pudo cargar la imagen {img_path}, se omite.")
            continue

        # Cargar landmarks
        landmarks = load_landmarks_from_txt(txt_path)
        if landmarks is None:
            print(f"No se pudo cargar landmarks desde {txt_path}, se omite.")
            continue

        # Dibujar landmarks sobre la imagen
        image_with_landmarks = draw_landmarks_on_image(image, landmarks)

        # Guardar en el directorio de salida
        output_path = os.path.join(output_dir, img_file)
        cv2.imwrite(output_path, image_with_landmarks)
        print(f"Guardada la imagen con landmarks en {output_path}")

# Ejemplo de uso:
image_dir = "/Users/jocareher/Downloads/baby_face_72_311/images"
txt_dir = "/Users/jocareher/Downloads/baby_face_72_311/labels"
output_dir = "/Users/jocareher/Downloads/plots_babyface72_311"
overlay_landmarks_on_images(image_dir, txt_dir, output_dir)


Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_289.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_276.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_262.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_302.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_316.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_100.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_114.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_128.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bcn_129.JPG
Guardada la imagen con landmarks en /Users/jocareher/Downloads/plots_babyface72_311/face_bc

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, List, Sequence, Tuple


def remap_visibility(visibility: int) -> int:
    """
    Remap landmark visibility labels from the original convention to the new one.

    Original convention:
        - 1: not visible
        - 2: visible

    New convention:
        - 0: not visible
        - 1: visible

    Args:
        visibility: Original visibility label.

    Returns:
        Remapped visibility label.

    Raises:
        ValueError: If the visibility label is not 1 or 2.
    """
    mapping = {1: 0, 2: 1}
    if visibility not in mapping:
        raise ValueError(
            f"Unexpected visibility label: {visibility}. Expected 1 or 2."
        )
    return mapping[visibility]


def parse_annotation_line(
    line: str,
    num_landmarks: int = 72,
) -> List[Tuple[float, float, int]]:
    """
    Parse a single annotation line and extract only landmark coordinates and visibility.

    Expected input format:
        class_idx cx cy w h x1 y1 v1 x2 y2 v2 ... xn yn vn

    Output format in memory:
        [(x1, y1, v1), (x2, y2, v2), ..., (xn, yn, vn)]

    Args:
        line: Raw annotation line from the input file.
        num_landmarks: Number of landmarks expected in the annotation.

    Returns:
        A list of tuples, each containing:
            - x coordinate (float)
            - y coordinate (float)
            - remapped visibility (int)

    Raises:
        ValueError: If the annotation length is inconsistent.
    """
    tokens = line.strip().split()
    if not tokens:
        return []

    expected_length = 5 + 3 * num_landmarks
    if len(tokens) != expected_length:
        raise ValueError(
            f"Invalid annotation length. Expected {expected_length} values, "
            f"but got {len(tokens)}. Line content: {line}"
        )

    landmark_tokens = tokens[5:]
    landmarks: List[Tuple[float, float, int]] = []

    for index in range(0, len(landmark_tokens), 3):
        x_coord = float(landmark_tokens[index])
        y_coord = float(landmark_tokens[index + 1])
        visibility = int(float(landmark_tokens[index + 2]))
        remapped = remap_visibility(visibility)
        landmarks.append((x_coord, y_coord, remapped))

    return landmarks


def format_landmarks_block(
    landmarks: Sequence[Tuple[float, float, int]],
    float_precision: int = 6,
) -> str:
    """
    Convert a list of landmarks into the target text block format.

    Output format:
        x1 y1 v1
        x2 y2 v2
        ...
        xn yn vn

    Args:
        landmarks: Sequence of landmark tuples (x, y, v).
        float_precision: Number of decimal places used for coordinates.

    Returns:
        A formatted text block for one face annotation.
    """
    lines = [
        f"{x:.{float_precision}f} {y:.{float_precision}f} {v}"
        for x, y, v in landmarks
    ]
    return "\n".join(lines)


def convert_label_file(
    input_file: Path,
    output_file: Path,
    num_landmarks: int = 72,
    float_precision: int = 6,
    separate_instances_with_blank_line: bool = True,
) -> None:
    """
    Convert a single label file from the original format to the new landmark-only format.

    If the input file contains multiple face annotations, each face is written as a
    separate block. By default, blocks are separated with one blank line.

    Args:
        input_file: Path to the original label file.
        output_file: Path where the converted label file will be written.
        num_landmarks: Number of landmarks expected per annotation.
        float_precision: Number of decimal places used for coordinates.
        separate_instances_with_blank_line: Whether to insert a blank line between
            multiple face annotations in the same file.

    Returns:
        None
    """
    raw_lines = input_file.read_text(encoding="utf-8").splitlines()

    blocks: List[str] = []
    for raw_line in raw_lines:
        if not raw_line.strip():
            continue

        landmarks = parse_annotation_line(raw_line, num_landmarks=num_landmarks)
        if landmarks:
            blocks.append(
                format_landmarks_block(
                    landmarks=landmarks,
                    float_precision=float_precision,
                )
            )

    output_file.parent.mkdir(parents=True, exist_ok=True)

    separator = "\n\n" if separate_instances_with_blank_line else "\n"
    output_text = separator.join(blocks)

    if output_text:
        output_text += "\n"

    output_file.write_text(output_text, encoding="utf-8")


def convert_split_labels(
    split_dir: Path,
    num_landmarks: int = 72,
    float_precision: int = 6,
    separate_instances_with_blank_line: bool = True,
    output_labels_dirname: str | None = None,
) -> None:
    """
    Convert all label files inside one dataset split.

    Expected split structure:
        split_dir/
            images/
            labels/

    By default, the original files inside 'labels/' are overwritten.
    If 'output_labels_dirname' is provided, converted files are written into:
        split_dir / output_labels_dirname

    Args:
        split_dir: Path to one split directory, such as train, val, or test.
        num_landmarks: Number of landmarks expected per annotation.
        float_precision: Number of decimal places used for coordinates.
        separate_instances_with_blank_line: Whether to insert blank lines between
            multiple face annotations in the same output file.
        output_labels_dirname: Optional new directory name for converted labels.
            If None, the original 'labels' directory is overwritten.

    Returns:
        None

    Raises:
        FileNotFoundError: If the labels directory does not exist.
    """
    input_labels_dir = split_dir / "labels"
    if not input_labels_dir.exists():
        raise FileNotFoundError(f"Labels directory not found: {input_labels_dir}")

    if output_labels_dirname is None:
        output_labels_dir = input_labels_dir
    else:
        output_labels_dir = split_dir / output_labels_dirname
        output_labels_dir.mkdir(parents=True, exist_ok=True)

    label_files = sorted(input_labels_dir.glob("*.txt"))

    for input_file in label_files:
        if output_labels_dir.resolve() == input_labels_dir.resolve():
            output_file = input_file
        else:
            output_file = output_labels_dir / input_file.name

        convert_label_file(
            input_file=input_file,
            output_file=output_file,
            num_landmarks=num_landmarks,
            float_precision=float_precision,
            separate_instances_with_blank_line=separate_instances_with_blank_line,
        )


def convert_dataset_labels(
    dataset_root: str | Path,
    splits: Sequence[str] = ("train", "val", "test"),
    num_landmarks: int = 72,
    float_precision: int = 6,
    separate_instances_with_blank_line: bool = True,
    output_labels_dirname: str | None = None,
) -> None:
    """
    Convert label files for all requested dataset splits.

    Expected dataset structure:
        dataset_root/
            train/
                images/
                labels/
            val/
                images/
                labels/
            test/
                images/
                labels/

    Args:
        dataset_root: Root directory of the dataset.
        splits: Split names to process.
        num_landmarks: Number of landmarks expected per annotation.
        float_precision: Number of decimal places used for coordinates.
        separate_instances_with_blank_line: Whether to insert blank lines between
            multiple face annotations in the same output file.
        output_labels_dirname: Optional new directory name for converted labels.
            If None, original label files are overwritten.

    Returns:
        None
    """
    dataset_root = Path(dataset_root)

    for split_name in splits:
        split_dir = dataset_root / split_name
        convert_split_labels(
            split_dir=split_dir,
            num_landmarks=num_landmarks,
            float_precision=float_precision,
            separate_instances_with_blank_line=separate_instances_with_blank_line,
            output_labels_dirname=output_labels_dirname,
        )


def preview_converted_label(
    input_file: str | Path,
    num_landmarks: int = 72,
    float_precision: int = 6,
    separate_instances_with_blank_line: bool = True,
) -> str:
    """
    Preview how one label file would look after conversion without writing it to disk.

    Args:
        input_file: Path to the original label file.
        num_landmarks: Number of landmarks expected per annotation.
        float_precision: Number of decimal places used for coordinates.
        separate_instances_with_blank_line: Whether to insert blank lines between
            multiple face annotations.

    Returns:
        Converted text content as a string.
    """
    input_file = Path(input_file)
    raw_lines = input_file.read_text(encoding="utf-8").splitlines()

    blocks: List[str] = []
    for raw_line in raw_lines:
        if not raw_line.strip():
            continue

        landmarks = parse_annotation_line(raw_line, num_landmarks=num_landmarks)
        if landmarks:
            blocks.append(
                format_landmarks_block(
                    landmarks=landmarks,
                    float_precision=float_precision,
                )
            )

    separator = "\n\n" if separate_instances_with_blank_line else "\n"
    return separator.join(blocks)

In [2]:
from pathlib import Path

sample_file = Path("/Users/jocareher/Documents/synthetic_lmks_vis_dataset/train/labels/synthetic_shape_00027_left.txt")
print(preview_converted_label(sample_file))

0.099518 0.411597 0
0.100589 0.406844 0
0.077582 0.390684 1
0.164259 0.401616 1
0.269663 0.388783 1
0.060995 0.604087 0
0.042804 0.573194 0
0.004815 0.563688 1
0.089353 0.575095 1
0.092028 0.605038 1
0.060460 0.612643 1
0.120920 0.729087 1
0.048154 0.685361 0
0.053505 0.690114 1
0.061530 0.685361 1
0.189941 0.737643 0
0.115035 0.782319 1
0.140717 0.815589 1
0.132691 0.904468 1
0.467095 0.552757 0
0.437667 0.680608 0
0.737293 0.568441 1
0.698234 0.727662 1
0.432317 0.745247 0
0.348315 0.792300 0
0.293740 0.824620 0
0.246656 0.857890 0
0.209738 0.895437 0
0.201177 0.921578 0
0.193151 0.975760 0
0.292670 0.944392 1
0.349385 0.930133 1
0.431247 0.904468 1
0.501338 0.878327 1
0.573034 0.847433 1
0.667202 0.801806 1
0.126271 0.400665 0
0.065811 0.352662 0
0.047084 0.314163 0
0.050294 0.307985 1
0.068486 0.347433 0
0.103264 0.341730 1
0.131086 0.290875 1
0.180310 0.288023 1
0.237025 0.322243 1
0.336544 0.376426 1
0.058855 0.465779 1
0.016586 0.511407 1
0.086142 0.373574 0
0.089888 0.371673 0


In [ ]:
convert_dataset_labels(
    dataset_root="/Users/jocareher/Documents/synthetic_lmks_vis_dataset",
    splits=("train", "val", "test"),
    num_landmarks=72,
    float_precision=6,
    separate_instances_with_blank_line=False,
    output_labels_dirname=None,
)